# 4장. pandas로 데이터에 질문하기

이 노트북은 `book/chapters/ch04_pandas_data_questions.md` 강의안을 초보자가 그대로 따라 하며 이해할 수 있도록 구성한 실습 자료입니다.

이번 장의 핵심은 복잡한 머신러닝이 아니라, 데이터를 정확히 선택하고, 조건에 맞게 걸러내고, 새 계산 컬럼을 만들고, 여러 파일을 연결해 요약표를 만드는 것입니다.


## 0. 이 노트북 사용 방법

아래 셀을 위에서부터 차례대로 실행하세요.

- 각 코드 셀은 바로 실행할 수 있게 구성했습니다.
- `customers.csv`, `products.csv`, `orders.csv`, `order_items.csv`가 `data/raw/` 폴더에 있어야 합니다.
- VS Code 또는 Jupyter에서 `notebooks/` 폴더 안의 이 파일을 실행하는 상황을 기준으로 경로를 잡았습니다.
- 오류가 나면 바로 위 설명 셀과 출력된 컬럼명을 먼저 확인하세요.


## 1. 이 장에서 답해 볼 질문

이번 장에서는 다음 질문에 pandas 코드로 답합니다.

1. 고객 데이터에서 필요한 컬럼만 선택하려면 어떻게 해야 할까?
2. 30세 이상 고객이나 특정 지역 고객만 추출하려면 어떻게 해야 할까?
3. 가격이 높은 상품이나 최근 주문을 빠르게 확인하려면 어떻게 정렬해야 할까?
4. 수량과 단가로 주문 상세 금액을 만들 수 있을까?
5. 카테고리별 매출, 상품별 매출, 월별 매출은 어떻게 계산할까?
6. 여러 CSV 파일을 연결할 때 무엇을 확인해야 할까?
7. LLM이 만들어 준 pandas 코드는 어떻게 검증해야 할까?


## 2. pandas 기본 분석 흐름

pandas 기본 분석은 보통 다음 순서로 진행됩니다.

| 단계 | pandas 기능 | 예시 |
|---|---|---|
| 컬럼 선택 | `df["컬럼명"]`, `df[[...]]` | 고객 ID와 나이만 선택 |
| 행 필터링 | 조건식 | 30세 이상 고객만 추출 |
| 정렬 | `sort_values()` | 가격이 높은 상품순 정렬 |
| 파생 컬럼 생성 | 새 컬럼 대입 | 수량 × 단가로 매출 계산 |
| 빈도 확인 | `value_counts()` | 지역별 고객 수 확인 |
| 그룹 집계 | `groupby()` | 카테고리별 매출 합계 |
| 파일 연결 | `merge()` | 주문 상세와 상품 데이터 연결 |
| 결과 저장 | `to_csv()` | 분석 요약 결과 CSV 저장 |


## 3. 기본 패키지와 경로 설정

먼저 필요한 패키지를 불러오고, 데이터 폴더와 결과 저장 폴더를 설정합니다.

`Path.cwd()`는 현재 노트북이 실행되는 위치입니다. 현재 위치가 `notebooks`라면 프로젝트 루트는 그 상위 폴더가 됩니다.


In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == 'notebooks':
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
REPORT_DIR = PROJECT_ROOT / 'reports'
REPORT_DIR.mkdir(exist_ok=True)

sns.set_theme(style='whitegrid')

print('현재 실행 위치:', CURRENT_DIR)
print('프로젝트 루트:', PROJECT_ROOT)
print('데이터 폴더:', DATA_DIR)
print('결과 저장 폴더:', REPORT_DIR)


## 4. 데이터 파일 불러오기

이번 장에서는 온라인 쇼핑몰 예제 데이터 4개를 사용합니다.

- `customers.csv`: 고객 정보
- `products.csv`: 상품 정보
- `orders.csv`: 주문 정보
- `order_items.csv`: 주문 상세 정보

파일이 없다는 오류가 나오면 먼저 샘플 데이터 생성 스크립트를 실행해야 합니다.


In [ ]:
customers = pd.read_csv(DATA_DIR / 'customers.csv')
products = pd.read_csv(DATA_DIR / 'products.csv')
orders = pd.read_csv(DATA_DIR / 'orders.csv')
order_items = pd.read_csv(DATA_DIR / 'order_items.csv')

print('데이터 불러오기 완료')


## 5. 데이터 크기와 컬럼명 확인하기

LLM이 작성한 pandas 코드가 틀리는 가장 흔한 이유는 실제 컬럼명과 다른 컬럼명을 사용하기 때문입니다.

분석을 시작하기 전에 항상 행과 열의 개수, 실제 컬럼명, 처음 몇 행의 값을 확인합니다.


In [ ]:
print('customers:', customers.shape, list(customers.columns))
print('products:', products.shape, list(products.columns))
print('orders:', orders.shape, list(orders.columns))
print('order_items:', order_items.shape, list(order_items.columns))


In [ ]:
customers.head()


In [ ]:
products.head()


In [ ]:
orders.head()


In [ ]:
order_items.head()


## 6. 필수 컬럼 존재 여부 점검하기

아래 코드는 이번 장 실습에서 사용할 컬럼들이 실제로 존재하는지 확인합니다.

누락된 컬럼이 있다면 데이터 생성 방식이 바뀌었거나 파일 버전이 다를 수 있습니다.


In [ ]:
expected_columns = {
    'customers': ['customer_id', 'gender', 'age', 'city'],
    'products': ['product_id', 'product_name', 'category', 'price'],
    'orders': ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status'],
    'order_items': ['order_id', 'product_id', 'quantity', 'unit_price'],
}

datasets = {
    'customers': customers,
    'products': products,
    'orders': orders,
    'order_items': order_items,
}

for name, cols in expected_columns.items():
    missing = [col for col in cols if col not in datasets[name].columns]
    if missing:
        print(f'{name}: 누락 컬럼 있음 -> {missing}')
    else:
        print(f'{name}: 필수 컬럼 모두 있음')


## 7. 컬럼 선택하기

컬럼 선택은 DataFrame에서 필요한 열만 가져오는 작업입니다.

- 컬럼 하나만 선택하면 `Series`가 됩니다.
- 컬럼 여러 개를 선택하면 `DataFrame`이 됩니다.
- 여러 컬럼을 선택할 때는 대괄호를 두 번 사용합니다. 예: `df[["col1", "col2"]]`


In [ ]:
customer_basic = customers[['customer_id', 'gender', 'age', 'city']]
customer_basic.head()


In [ ]:
city_series = customers['city']

print(type(city_series))
city_series.head()


In [ ]:
city_age_dataframe = customers[['city', 'age']]

print(type(city_age_dataframe))
city_age_dataframe.head()


In [ ]:
product_basic = products[['product_id', 'product_name', 'category', 'price']]
product_basic.head()


## 8. 행 필터링하기

행 필터링은 조건에 맞는 데이터만 추출하는 작업입니다.

예를 들어 `customers["age"] >= 30`은 각 행이 조건을 만족하는지 `True/False`로 판단합니다. 이 결과를 다시 `customers[...]` 안에 넣으면 조건을 만족하는 행만 남습니다.


In [ ]:
customers_over_30 = customers[customers['age'] >= 30]

print('전체 고객 수:', len(customers))
print('30세 이상 고객 수:', len(customers_over_30))

customers_over_30.head()


### 도시별 고객 수 확인하기

특정 지역 고객을 필터링하기 전에는 실제 데이터에 어떤 도시명이 들어 있는지 먼저 확인해야 합니다. 예를 들어 데이터에는 `Seoul`이 아니라 `서울`로 들어 있을 수도 있습니다.


In [ ]:
customers['city'].value_counts().head(10)


In [ ]:
seoul_customers = customers[customers['city'] == 'Seoul']

print('서울 고객 수:', len(seoul_customers))
seoul_customers.head()


### 조건이 여러 개인 경우

조건이 여러 개일 때는 `&`, `|`, `~`를 사용합니다.

| 연산자 | 의미 | 예시 |
|---|---|---|
| `&` | 그리고 | 30세 이상이면서 서울 거주 |
| `|` | 또는 | 서울 또는 부산 거주 |
| `~` | 아니다 | 완료 상태가 아닌 주문 |

중요: 각 조건은 반드시 괄호로 감싸는 습관을 들이세요.


In [ ]:
target_customers = customers[
    (customers['age'] >= 30) &
    (customers['city'] == 'Seoul')
]

print('조건에 맞는 고객 수:', len(target_customers))
target_customers.head()


In [ ]:
city_customers = customers[customers['city'].isin(['Seoul', 'Busan'])]

print('서울 또는 부산 고객 수:', len(city_customers))
city_customers.head()


## 9. 주문 상태와 정렬 확인하기

`value_counts()`는 특정 컬럼에 어떤 값이 몇 번 나오는지 확인할 때 사용합니다. 필터링 전에 실제 값의 종류를 먼저 확인하면 잘못된 조건식을 줄일 수 있습니다.


In [ ]:
orders['order_status'].value_counts()


In [ ]:
completed_orders = orders[orders['order_status'] == 'completed']

print('전체 주문 수:', len(orders))
print('완료 주문 수:', len(completed_orders))

completed_orders.head()


### 정렬하기

`sort_values()`는 특정 컬럼을 기준으로 데이터를 정렬합니다.

- `ascending=True`: 오름차순, 작은 값부터
- `ascending=False`: 내림차순, 큰 값부터

가격, 매출, 주문 수처럼 큰 값을 먼저 보고 싶을 때는 `ascending=False`를 자주 사용합니다.


In [ ]:
top_price_products = products.sort_values('price', ascending=False)
top_price_products.head(10)


In [ ]:
customers.sort_values('age', ascending=False).head(10)


## 10. 파생 컬럼 만들기

파생 컬럼은 기존 컬럼을 계산해서 새로 만든 컬럼입니다. 주문 상세 데이터에는 보통 `quantity`와 `unit_price`가 있습니다. 두 값을 곱하면 주문 상세 1행의 금액을 계산할 수 있습니다.

`line_total = quantity × unit_price`


In [ ]:
order_items = order_items.copy()

order_items['line_total'] = order_items['quantity'] * order_items['unit_price']

order_items[['order_id', 'product_id', 'quantity', 'unit_price', 'line_total']].head()


In [ ]:
order_items['line_total'].describe()


In [ ]:
total_sales = order_items['line_total'].sum()

print('주문 상세 기준 전체 금액 합계:', total_sales)


`total_sales`는 주문 상세 기준의 전체 금액 합계입니다. 다만 이 값에 취소 주문이나 환불 주문이 포함되어 있는지는 별도로 확인해야 합니다. 실제 업무에서는 주문 상태와 결제 상태를 함께 확인하는 과정이 필요합니다.


## 11. `merge()`로 여러 파일 연결하기

온라인 쇼핑몰 데이터는 하나의 파일만으로 충분히 분석하기 어렵습니다. 카테고리별 매출을 계산하려면 주문 상세 데이터와 상품 데이터를 연결해야 합니다.

- `order_items`: 주문별 상품, 수량, 단가
- `products`: 상품명, 카테고리, 가격

두 데이터에는 공통으로 `product_id`가 있으므로 이 컬럼을 기준으로 연결할 수 있습니다.


In [ ]:
before_rows = len(order_items)

sales_items = order_items.merge(
    products,
    on='product_id',
    how='left'
)

after_rows = len(sales_items)

print('병합 전 order_items 행 수:', before_rows)
print('병합 후 sales_items 행 수:', after_rows)
print('행 수가 같은가?', before_rows == after_rows)

sales_items.head()


### 병합 결과 검증하기

`how="left"`를 사용했다면 일반적으로 왼쪽 데이터인 `order_items`의 행 수가 유지되어야 합니다. 병합 후에는 행 수, 누락값, 중복 컬럼을 확인해야 합니다.


In [ ]:
print('product_name 누락 수:', sales_items['product_name'].isna().sum())
print('category 누락 수:', sales_items['category'].isna().sum())

list(sales_items.columns)


## 12. 카테고리별 매출 집계하기

`groupby()`는 데이터를 특정 기준으로 묶은 뒤 합계, 평균, 개수 등을 계산하는 기능입니다. 카테고리별 매출은 `category` 기준으로 묶고 `line_total` 합계를 계산하면 됩니다.


In [ ]:
category_sales = (
    sales_items
    .groupby('category', as_index=False)['line_total']
    .sum()
    .rename(columns={'line_total': 'total_sales'})
    .sort_values('total_sales', ascending=False)
)

category_sales['sales_ratio'] = (
    category_sales['total_sales'] / category_sales['total_sales'].sum() * 100
)

category_sales


카테고리별 매출 결과를 읽을 때는 매출이 높은 이유가 판매 수량 때문인지, 단가 때문인지 구분해야 합니다. 이 결과만으로 원인을 단정하지 말고 추가 분석 질문을 만들어야 합니다.


## 13. 상품별 매출 집계하기

상품별 매출은 어떤 상품이 많이 팔렸는지 확인하는 데 사용합니다. 여기서는 상품별 수량 합계와 매출 합계를 함께 계산합니다.


In [ ]:
product_sales = (
    sales_items
    .groupby(['product_id', 'product_name', 'category'], as_index=False)
    .agg(
        total_quantity=('quantity', 'sum'),
        total_sales=('line_total', 'sum')
    )
    .sort_values('total_sales', ascending=False)
)

product_sales.head(10)


## 14. 월별 매출 집계하기

월별 매출을 계산하려면 주문일 정보가 필요합니다. 매출 금액은 `order_items`에 있고 주문일은 `orders`에 있으므로 두 데이터를 `order_id` 기준으로 연결합니다.


In [ ]:
order_sales = order_items.merge(
    orders,
    on='order_id',
    how='left'
)

print('order_date 누락 수:', order_sales['order_date'].isna().sum())
order_sales.head()


In [ ]:
order_sales['order_date'] = pd.to_datetime(order_sales['order_date'], errors='coerce')

print('날짜 변환 실패 수:', order_sales['order_date'].isna().sum())

order_sales['order_month'] = order_sales['order_date'].dt.to_period('M').astype(str)

order_sales[['order_id', 'order_date', 'order_month', 'line_total']].head()


In [ ]:
monthly_summary = (
    order_sales
    .groupby('order_month', as_index=False)
    .agg(
        total_sales=('line_total', 'sum'),
        order_count=('order_id', 'nunique')
    )
    .sort_values('order_month')
)

monthly_summary


`agg(새컬럼명=(원본컬럼명, 집계함수))` 형식은 집계 결과 컬럼명을 직접 정하는 방법입니다. 여기서 `order_count=("order_id", "nunique")`는 주문 상세 행 수가 아니라 고유한 주문 건수를 계산한다는 뜻입니다.


## 15. 고객별 구매 금액 집계하기

고객별 구매 금액을 계산하려면 `order_items`, `orders`, `customers`를 연결해야 합니다. 개인정보가 포함될 수 있으므로 보고서에서는 고객 실명, 이메일, 전화번호를 직접 노출하지 않도록 주의합니다.


In [ ]:
customer_sales_base = order_sales.merge(
    customers,
    on='customer_id',
    how='left'
)

customer_sales_base.head()


In [ ]:
group_columns = ['customer_id', 'city']
if 'name' in customer_sales_base.columns:
    group_columns = ['customer_id', 'name', 'city']

customer_sales = (
    customer_sales_base
    .groupby(group_columns, as_index=False)
    .agg(
        order_count=('order_id', 'nunique'),
        total_sales=('line_total', 'sum')
    )
    .sort_values('total_sales', ascending=False)
)

customer_sales.head(10)


## 16. 간단한 시각화로 결과 확인하기

4장의 중심은 pandas 기본 분석이지만, 요약표를 그래프로 확인하면 결과를 더 쉽게 이해할 수 있습니다.


In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(data=category_sales, x='category', y='total_sales')
plt.title('카테고리별 매출')
plt.xlabel('카테고리')
plt.ylabel('매출 합계')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 5))
sns.lineplot(data=monthly_summary, x='order_month', y='total_sales', marker='o')
plt.title('월별 매출 추이')
plt.xlabel('월')
plt.ylabel('매출 합계')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 17. 분석 결과 저장하기

분석 결과를 CSV 파일로 저장하면 보고서 작성, 과제 제출, 후속 분석에 활용할 수 있습니다. Windows Excel에서 한글이 포함된 CSV를 바로 열 계획이라면 `encoding="utf-8-sig"` 옵션을 사용하는 것이 안전합니다.


In [ ]:
category_sales.to_csv(REPORT_DIR / 'ch04_category_sales.csv', index=False, encoding='utf-8-sig')
product_sales.to_csv(REPORT_DIR / 'ch04_product_sales.csv', index=False, encoding='utf-8-sig')
monthly_summary.to_csv(REPORT_DIR / 'ch04_monthly_sales.csv', index=False, encoding='utf-8-sig')
customer_sales.to_csv(REPORT_DIR / 'ch04_customer_sales.csv', index=False, encoding='utf-8-sig')

print('저장 완료')
print(REPORT_DIR / 'ch04_category_sales.csv')
print(REPORT_DIR / 'ch04_product_sales.csv')
print(REPORT_DIR / 'ch04_monthly_sales.csv')
print(REPORT_DIR / 'ch04_customer_sales.csv')


## 18. 반복되는 데이터 점검을 함수로 정리하기

분석을 하다 보면 데이터 크기, 컬럼명, 결측치, 중복 행을 반복해서 확인하게 됩니다. 이런 작업은 함수로 만들어 두면 편리합니다.


In [ ]:
def summarize_dataframe(name, df):
    print(f'===== {name} =====')
    print('shape:', df.shape)
    print('columns:', list(df.columns))
    print('missing values:', df.isna().sum().sum())
    print('duplicated rows:', df.duplicated().sum())


In [ ]:
for name, df in datasets.items():
    summarize_dataframe(name, df)
    print()


## 19. LLM에게 pandas 코드를 요청하는 방법

LLM에게 코드를 요청할 때는 실제 원본 데이터를 그대로 넣기보다 DataFrame 이름, 컬럼명, 하고 싶은 작업, 원하는 결과 형태를 정리해서 전달하는 것이 좋습니다.

```text
다음 customers DataFrame에서 조건에 맞는 데이터를 필터링하는 pandas 코드를 작성해 주세요.

DataFrame 이름: customers
컬럼: customer_id, name, gender, age, city, signup_date
조건: age가 30 이상이고 city가 Seoul 또는 Busan

초보자가 이해할 수 있도록 코드와 설명을 함께 작성해 주세요.
단, 실제 데이터가 아니라 컬럼 구조만 보고 작성해 주세요.
```


## 20. LLM 코드 검증 예시

LLM이 다음과 같은 코드를 제안했다고 가정해 보겠습니다.

```python
category_sales = order_items.groupby("category")["line_total"].sum()
```

이 코드는 그럴듯해 보이지만, 현재 데이터 구조에서는 바로 실행되지 않을 가능성이 큽니다. `category` 컬럼은 `order_items`가 아니라 `products`에 있기 때문입니다. 따라서 먼저 `order_items`와 `products`를 `product_id` 기준으로 병합해야 합니다.


In [ ]:
print('order_items 컬럼:', list(order_items.columns))
print('products 컬럼:', list(products.columns))

print('order_items에 category가 있나요?', 'category' in order_items.columns)
print('products에 category가 있나요?', 'category' in products.columns)


In [ ]:
correct_category_sales = (
    order_items
    .merge(products, on='product_id', how='left')
    .groupby('category', as_index=False)['line_total']
    .sum()
    .rename(columns={'line_total': 'total_sales'})
    .sort_values('total_sales', ascending=False)
)

correct_category_sales


## 21. 결과 해석 연습

카테고리별 매출 결과를 해석할 때는 데이터에 없는 원인을 단정하지 않아야 합니다.

좋은 해석 예시:

> 전자기기 카테고리의 매출 비중이 가장 높게 나타났다. 다만 이 결과만으로 전자기기 수요가 증가했다고 단정할 수는 없다. 판매 수량, 평균 단가, 할인 여부, 주문 취소 여부를 추가로 확인할 필요가 있다.

나쁜 해석 예시:

> 전자기기 매출이 높은 이유는 고객들이 전자기기를 가장 좋아하기 때문이다.

두 번째 문장은 데이터에 없는 심리적 원인을 단정하고 있으므로 주의해야 합니다.


## 22. 실습 과제

아래 과제를 직접 해결해 보세요.

1. 40세 이상 고객만 추출해 `customers_over_40` 변수에 저장하세요.
2. 상품 가격이 낮은 순서대로 상위 10개 상품을 출력하세요.
3. 결제수단별 주문 수를 `value_counts()`로 확인하세요.
4. 카테고리별 평균 상품 가격을 계산하세요.
5. 월별 주문 건수와 매출을 함께 계산하세요.
6. LLM에게 고객별 구매 금액 집계 코드를 요청하는 프롬프트를 직접 작성해 보세요.


In [ ]:
# 과제 1. 40세 이상 고객만 추출하세요.
# customers_over_40 = ...


In [ ]:
# 과제 2. 상품 가격이 낮은 순서대로 상위 10개 상품을 출력하세요.
# low_price_products = ...


In [ ]:
# 과제 3. 결제수단별 주문 수를 확인하세요.
# orders['payment_method'].value_counts()


In [ ]:
# 과제 4. 카테고리별 평균 상품 가격을 계산하세요.
# category_price_mean = ...


In [ ]:
# 과제 5. 월별 주문 건수와 매출을 함께 계산하세요.
# monthly_order_sales = ...


## 23. 정리

이번 장에서는 pandas로 다음 작업을 수행했습니다.

- 필요한 컬럼 선택
- 조건에 맞는 행 필터링
- 값의 빈도 확인
- 정렬
- 파생 컬럼 생성
- `merge()`로 여러 파일 연결
- `groupby()`로 카테고리별, 상품별, 월별, 고객별 요약표 생성
- CSV 저장
- LLM 코드 검증

다음 장에서는 이런 분석을 더 믿을 수 있게 만들기 위해 결측치, 중복, 타입 오류, 날짜 형식, 이상값 후보를 정리하는 데이터 전처리를 다룹니다.
